# 5.4 · 多服务器宿主：渐进式发现与代码模式

**Multi-server hosts: progressive discovery and code mode**

> **学完本课你能：** 设计一个聚合多个 MCP 服务器又不会把模型淹没的宿主——命名空间路由、渐进式工具发现、遵循 TTL 的缓存，以及由宿主代理的代码模式执行。

官方阅读材料：[Client best practices](https://modelcontextprotocol.io/docs/2026-07-28/develop/clients/client-best-practices) · [Understanding MCP servers (multi-server example)](https://modelcontextprotocol.io/docs/2026-07-28/learn/server-concepts)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 三个服务器，一次旅行
这是文档中的旅行规划示例，缩小到适合笔记本的规模：一个 <strong>flights</strong> 服务器、一个 <strong>hotels</strong> 服务器和一个 <strong>weather</strong> 服务器，全部在进程内运行。有两个细节是刻意设计的。两个旅行服务器都暴露了名为 `search` 的工具（工具名只在服务器<em>内部</em>唯一），并且两者报告的 `serverInfo.name` 相同，都是 `"travel-api"`（规范指出不要依赖这个名称来区分服务器（Server））。flights 服务器还会返回大量数据，这一点在代码模式（Code Mode）中会很重要。

In [ ]:
import ast
import asyncio
import hashlib
import json
import logging
import os
import random
import time

from mcp import Client
from mcp.server import MCPServer
from mcp.server.caching import CacheHint
from mcp.server.mcpserver.exceptions import ToolError

logging.getLogger("mcp").setLevel(logging.WARNING)

rng = random.Random(7)
AIRLINES = ["IB", "VY", "UA", "DL", "AA", "BA", "LH", "AF"]
FLIGHTS = [{"id": f"{rng.choice(AIRLINES)}{rng.randint(100, 9999)}", "origin": "NYC", "destination": "BCN",
            "date": "2026-06-15", "depart": f"{rng.randint(6, 22):02d}:{rng.choice(['00', '15', '30', '45'])}",
            "stops": rng.choice([0, 0, 1, 1, 2]), "price": rng.randint(380, 1400),
            "fare_rules": "Non-refundable. Changes allowed for a fee. Seat selection extra."}
           for _ in range(40)]
HOTELS = [{"name": n, "city": "Barcelona", "price": p, "rating": r}
          for n, p, r in [("Casa Gracia", 140, 4.6), ("Hotel Arts", 420, 4.8), ("Generator", 70, 4.1),
                          ("Praktik Rambla", 165, 4.5), ("H10 Metropolitan", 210, 4.4)]]

# One vendor runs both travel servers, so serverInfo.name is the same: "travel-api".
flights = MCPServer("travel-api", cache_hints={"tools/list": CacheHint(ttl_ms=300_000, scope="public")})
hotels = MCPServer("travel-api", cache_hints={"tools/list": CacheHint(ttl_ms=300_000, scope="public")})
weather = MCPServer("weather", cache_hints={"tools/list": CacheHint(ttl_ms=60_000, scope="public")})


@flights.tool()
def search(origin: str, destination: str, date: str) -> list[dict]:
    """Search flights between two airports on a date (YYYY-MM-DD). Returns every matching flight."""
    return [f for f in FLIGHTS if (f["origin"], f["destination"], f["date"]) == (origin, destination, date)]


@hotels.tool(name="search")
def search_hotels(city: str, max_price: int) -> list[dict]:
    """Search hotels in a city with a nightly price at or below max_price (USD)."""
    return [h for h in HOTELS if h["city"] == city and h["price"] <= max_price]


@hotels.tool()
def book(name: str, nights: int) -> str:
    """Book a hotel room for a number of nights. Has side effects: charges the user's card."""
    return f"booked {name} for {nights} nights"


@weather.tool()
def forecast(city: str, date: str) -> str:
    """Daily weather forecast for a city on a date (YYYY-MM-DD)."""
    if city != "Barcelona":
        raise ToolError(f"no forecast data for {city!r}")  # ToolError: the message reaches the client
    return "sunny, 27°C, light sea breeze"


print(len(FLIGHTS), "flights,", len(HOTELS), "hotels; sample flight:", json.dumps(FLIGHTS[0]))

### 2. 多服务器宿主：每个服务器一个客户端，工具加命名空间
宿主（Host）为每个服务器持有一个 `Client`，以<strong>它自己的配置名</strong>（`flights`、`hotels`、`weather`）为键，而不是 `serverInfo.name`。它给每个工具加上该键作为前缀（`flights__search`），这样既能保证名称在各服务器之间唯一，又仍在允许的工具名字符范围内；路由调用时再把前缀拆掉。客户端（Client）是惰性连接的，所以服务器在被需要之前不产生任何开销（动态服务器管理，Dynamic Server Management），而 `disable()` 会断开它。

宿主还维护着<strong>自己的工具定义缓存</strong>，遵循 `ttlMs`（SDK 的 `Client` 也有内置缓存；我们调用 `list_tools(cache_mode="bypass")`，确保你在这里看到的每次获取都真正发往服务器）。

In [ ]:
SEP = "__"


class Host:
    def __init__(self, registry: dict[str, dict], clock=time.monotonic):
        self.registry = registry          # alias -> {"server": ..., "about": "..."}
        self.clock = clock
        self.clients: dict[str, Client] = {}
        self.owners: dict[str, tuple[asyncio.Task, asyncio.Event]] = {}
        self.cache: dict[str, dict] = {}  # alias -> {"tools", "received", "ttl_ms", "scope"}
        self.list_requests = 0

    async def _own(self, alias: str, ready: asyncio.Event, stop: asyncio.Event) -> None:
        # One long-lived task per connection: it enters and exits the Client in the same task.
        async with Client(self.registry[alias]["server"]) as client:
            self.clients[alias] = client
            ready.set()
            await stop.wait()

    async def enable(self, alias: str) -> Client:
        if alias not in self.clients:
            ready, stop = asyncio.Event(), asyncio.Event()
            task = asyncio.create_task(self._own(alias, ready, stop))
            self.owners[alias] = (task, stop)
            await asyncio.wait([task, asyncio.create_task(ready.wait())], return_when=asyncio.FIRST_COMPLETED)
            if task.done():                # the connection failed before it was ready
                self.owners.pop(alias)
                raise task.exception()
        return self.clients[alias]

    async def disable(self, alias: str) -> None:
        self.cache.pop(alias, None)
        self.clients.pop(alias, None)
        task, stop = self.owners.pop(alias)
        stop.set()
        await task

    async def close(self) -> None:
        for alias in list(self.owners):
            await self.disable(alias)

    async def tools_of(self, alias: str):
        entry = self.cache.get(alias)
        if entry and self.clock() < entry["received"] + entry["ttl_ms"] / 1000:   # now < t_received + ttlMs
            return entry["tools"]
        client = await self.enable(alias)
        result = await client.list_tools(cache_mode="bypass")
        self.list_requests += 1
        self.cache[alias] = {"tools": result.tools, "received": self.clock(),
                             "ttl_ms": max(result.ttl_ms or 0, 0), "scope": result.cache_scope}
        return result.tools

    def on_list_changed(self, alias: str) -> None:
        """notifications/tools/list_changed: stale immediately, even inside the TTL."""
        self.cache.pop(alias, None)

    async def definitions(self, aliases=None) -> list[dict]:
        """Namespaced definitions in the shape a model API expects, grouped by server, stable order."""
        out = []
        for alias in aliases or self.registry:
            for t in await self.tools_of(alias):
                out.append({"name": f"{alias}{SEP}{t.name}", "description": t.description or "",
                            "input_schema": t.input_schema})
        return out

    async def call(self, name: str, arguments: dict):
        alias, sep, tool = name.partition(SEP)
        if not sep or alias not in self.registry:
            raise KeyError(f"unknown tool {name!r}")
        client = await self.enable(alias)
        return await client.call_tool(tool, arguments)


REGISTRY = {
    "flights": {"server": flights, "about": "Flight search across airlines"},
    "hotels": {"server": hotels, "about": "Hotel search and booking"},
    "weather": {"server": weather, "about": "Weather forecasts"},
}
host = Host(REGISTRY)
for d in await host.definitions():
    print(f"{d['name']:<18} {d['description'][:70]}")
print("\nconnected:", sorted(host.clients), "| serverInfo names:",
      [c.server_info.name for c in host.clients.values()])
result = await host.call("hotels__search", {"city": "Barcelona", "max_price": 150})
print("hotels__search ->", result.structured_content)

如果没有前缀，宿主（Host）就会有两个名为 `search` 的工具，并且无从知道模型指的是哪个服务器。按服务器分组（所有 `flights__*` 放在一起）也有助于模型推理相关的能力。

### 3. “全部加载”要花多少？
你在 `tools` 数组中传入的每个定义，在<strong>每一次</strong>模型调用中都是提示词令牌（Token）。我们使用粗略的经验法则 <em>字符数 ÷ 4</em>。三个小服务器很便宜，所以再加入第四个贴近现实的服务器：一个拥有 120 个工具的 CRM 服务器，真实的 SaaS 服务器提供的正是这种规模的目录。

In [ ]:
def est_tokens(obj) -> int:
    return len(json.dumps(obj)) // 4


crm = MCPServer("crm")
OBJECTS = ["account", "contact", "lead", "opportunity", "case", "campaign", "quote", "invoice",
           "order", "product", "task", "note", "contract", "asset", "survey"]
VERBS = {"get": "Fetch one {o} record by id, including all standard and custom fields.",
         "list": "List {o} records matching a filter expression, paginated, newest first.",
         "create": "Create a new {o} record from a map of field names to values.",
         "update": "Update fields on an existing {o} record; unspecified fields are unchanged.",
         "delete": "Permanently delete a {o} record. Cannot be undone.",
         "search": "Full-text search over {o} records; returns ids and display names.",
         "export": "Export {o} records matching a filter to CSV and return a download link.",
         "history": "Return the audit history (who changed what, when) for one {o} record."}


def crm_tool(record_id: str, fields: dict | None = None, filter: str | None = None, limit: int = 50) -> str:
    return "ok"


for o in OBJECTS:
    for verb, text in VERBS.items():
        crm.add_tool(crm_tool, name=f"{o}_{verb}", description=text.format(o=o))

REGISTRY["crm"] = {"server": crm, "about": "Customer records: accounts, contacts, leads, opportunities, cases"}

CONTEXT_WINDOW = 200_000
THRESHOLD = 0.02                       # the docs suggest a 1%–5% threshold
all_defs = await host.definitions()
for alias in REGISTRY:
    print(f"{alias:<8} {sum(d['name'].startswith(alias + SEP) for d in all_defs):>4} tools "
          f"~{est_tokens([d for d in all_defs if d['name'].startswith(alias + SEP)]):>6} tokens")
total = est_tokens(all_defs)
print(f"\nall {len(all_defs)} definitions ~{total} tokens = {total / CONTEXT_WINDOW:.1%} of a "
      f"{CONTEXT_WINDOW:,}-token window -> "
      f"{'switch to progressive discovery' if total > THRESHOLD * CONTEXT_WINDOW else 'just load them all'}")

### 4. 渐进式发现：目录 → 查看 → 执行
模型拿到的不再是 120 多个定义，而是三个小小的<strong>元工具（Meta-tool）</strong>。`search_tools` 返回名称和一行描述（第 1 层，目录），`get_tool_details` 返回单个完整模式（第 2 层，查看），`call_tool` 按名称执行任何已发现的工具（第 3 层，执行）。这里的搜索只是简单的关键词打分；嵌入向量或小模型也可以放进同一个函数中。

In [ ]:
META_TOOLS = [
    {"name": "search_tools", "description": "Search all connected MCP tools. Returns names and one-line "
     "descriptions grouped by server. Use detail='names' for just names.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string"},
                      "detail": {"type": "string", "enum": ["names", "summary"]}}, "required": ["query"]}},
    {"name": "get_tool_details", "description": "Full definition (input schema) of one tool by name.",
     "input_schema": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]}},
    {"name": "call_tool", "description": "Call a tool found with search_tools, by its full name.",
     "input_schema": {"type": "object", "properties": {"name": {"type": "string"},
                      "arguments": {"type": "object"}}, "required": ["name", "arguments"]}},
]


async def search_tools(query: str, detail: str = "summary", k: int = 5):
    words = {w.strip(".,").lower() for w in query.split()}
    scored = []
    for d in await host.definitions():
        text = (d["name"].replace(SEP, " ").replace("_", " ") + " " + d["description"]).lower()
        score = sum(w in text for w in words)
        if score:
            scored.append((-score, d["name"], d["description"].split(". ")[0]))
    hits = sorted(scored)[:k]
    return [h[1] for h in hits] if detail == "names" else [{"name": n, "description": s} for _, n, s in hits]


async def get_tool_details(name: str):
    return next(d for d in await host.definitions() if d["name"] == name)


async def run_meta_tool(name: str, args: dict):
    if name == "search_tools":
        return await search_tools(**args)
    if name == "get_tool_details":
        return await get_tool_details(**args)
    if name == "call_tool":
        r = await host.call(args["name"], args["arguments"])
        return {"isError": r.is_error, "content": r.structured_content or r.content[0].text}
    raise KeyError(name)


# A scripted "model" that plans a trip using only the three meta-tools.
script = [("search_tools", {"query": "weather forecast city"}),
          ("get_tool_details", {"name": "weather__forecast"}),
          ("call_tool", {"name": "weather__forecast", "arguments": {"city": "Barcelona", "date": "2026-06-15"}})]
context = est_tokens(META_TOOLS)
print(f"meta-tool definitions: ~{context} tokens (vs ~{total} for everything)\n")
for name, args in script:
    out = await run_meta_tool(name, args)
    context += est_tokens(out)
    print(f">>> {name}({json.dumps(args)})\n<<< {json.dumps(out)[:300]}\n")
print(f"context spent on tools after the task: ~{context} tokens ({context / total:.1%} of loading everything)")

### 5. 缓存：`ttlMs`、`list_changed` 与提示缓存陷阱
这里涉及两种不同的缓存。<strong>宿主侧定义缓存</strong>遵循服务器的 `ttlMs`（新鲜度：`now < t_received + ttlMs`），并会因 `notifications/tools/list_changed` 而失效（Invalidation）。<strong>模型提供方的提示缓存（Prompt Caching）</strong>覆盖请求的前缀，其中包括 `tools` 数组；一旦修改这个数组，缓存就会未命中。我们使用一个假时钟，免得大家干等五分钟。

In [ ]:
class FakeClock:
    def __init__(self):
        self.t = 1000.0

    def __call__(self):
        return self.t


clock = FakeClock()
cached_host = Host({k: REGISTRY[k] for k in ("flights", "weather")}, clock=clock)


async def need_tools(label):
    before = cached_host.list_requests
    await cached_host.definitions()
    hit = cached_host.list_requests == before
    print(f"{label:<46} tools/list sent: {cached_host.list_requests - before}  ({'cache hit' if hit else 'fetched'})")


await need_tools("t=0      first use")
print("          cached:", {a: (e["ttl_ms"], e["scope"]) for a, e in cached_host.cache.items()})
clock.t += 30
await need_tools("t=30s    both fresh")
clock.t += 45
await need_tools("t=75s    weather TTL (60 s) expired")
cached_host.on_list_changed("flights")
await need_tools("list_changed from flights (TTL still fresh)")
await cached_host.close()

再看提示缓存（Prompt Caching）这一侧。我们在两种策略下为模型每一轮看到的 `tools` 数组计算指纹：一种是把已发现的定义注入 `tools`（并重新排序），另一种是只保留稳定的元工具（Meta-tool），所有调用都经由 `call_tool` 路由。

In [ ]:
def fingerprint(tools):
    return hashlib.sha256(json.dumps(tools, sort_keys=True).encode()).hexdigest()[:10]


weather_def = await get_tool_details("weather__forecast")
hotel_def = await get_tool_details("hotels__search")
turns_inject = [META_TOOLS, sorted(META_TOOLS + [weather_def], key=lambda d: d["name"]),
                sorted(META_TOOLS + [weather_def, hotel_def], key=lambda d: d["name"])]
turns_stable = [META_TOOLS, META_TOOLS, META_TOOLS]
for i, (a, b) in enumerate(zip(turns_inject, turns_stable), 1):
    print(f"turn {i}: inject+sort tools={fingerprint(a)} | stable call_tool tools={fingerprint(b)}")
print("\nA changed fingerprint means the provider's cached prefix no longer matches -> cache miss.")
print("Fix: append new definitions after the cache breakpoint, or keep the array fixed and use call_tool.")

### 6. 代码模式：从模式生成带类型的 API
为了实现程序化工具调用（Programmatic Tool Calling），宿主把每个工具的模式（Schema）转换为模型可以据此编码的函数签名。放进提示词里的就是这一小段文本，而不是 JSON 模式。

In [ ]:
PY_TYPES = {"string": "str", "integer": "int", "number": "float", "boolean": "bool", "object": "dict",
            "array": "list"}


def stub_source(d: dict) -> str:
    props, required = d["input_schema"].get("properties", {}), d["input_schema"].get("required", [])
    params = []
    for p, s in props.items():
        t = PY_TYPES.get(s.get("type"), "object")
        params.append(f"{p}: {t}" if p in required else f"{p}: {t} | None = None")
    return f'async def {d["name"]}({", ".join(params)}):\n    """{d["description"]}"""'


travel_defs = await host.definitions(["flights", "hotels", "weather"])
api_text = "\n\n".join(stub_source(d) for d in travel_defs)
print(api_text)
print(f"\n~{est_tokens(api_text)} tokens of API description")

### 7. 玩具执行器与逐次调用授权的代理
执行器运行模型编写的代码时，作用域中只有生成的函数和 `print`。每个生成的函数都是一个<strong>代理桩函数（Broker Stub）</strong>：它检查本次运行的审批授权，把调用以 `tools/call` 转发给正确的服务器，并把 `isError: true` 转换为脚本可以 `try`/`except` 的异常。

> <strong>这不是沙箱。</strong> 带着精简过的 `__builtins__` 和 AST 过滤器的 `exec` 只是一种教学手段；CPython 有很多逃逸途径，而且这里没有内存限制。真正的宿主会在独立的隔离运行时中运行模型代码（例如 Deno 或 `isolated-vm` 这样的 V8 隔离环境、Wasmtime 这样的 Wasm 运行时、Monty 这样的极简解释器，或是锁定的容器／微虚拟机），<strong>没有网络</strong>，有超时和内存限制，且只有一条通往代理的消息通道。

In [ ]:
class ScriptToolError(Exception):
    """What a generated stub raises inside the script when the server answers isError: true."""


SAFE_BUILTINS = {n: __builtins__[n] if isinstance(__builtins__, dict) else getattr(__builtins__, n)
                 for n in ["len", "min", "max", "sum", "sorted", "range", "enumerate", "zip", "round",
                           "str", "int", "float", "dict", "list", "set", "any", "all", "Exception"]}


def check_script(code: str) -> None:
    for node in ast.walk(ast.parse(code)):
        if isinstance(node, (ast.Import, ast.ImportFrom, ast.Global, ast.Nonlocal)):
            raise PermissionError(f"line {node.lineno}: imports/globals are not allowed")
        if isinstance(node, ast.Attribute) and node.attr.startswith("_"):
            raise PermissionError(f"line {node.lineno}: private attribute {node.attr!r} is not allowed")
        if isinstance(node, ast.Name) and node.id.startswith("__"):
            raise PermissionError(f"line {node.lineno}: dunder name {node.id!r} is not allowed")


async def run_script(code: str, grant: set[str], timeout: float = 5.0, max_output: int = 1000) -> dict:
    """Run model-written code; return only what it prints (truncated) plus a call log for the host."""
    calls, out = [], []

    def make_stub(name):
        async def stub(**kwargs):
            if name not in grant:                       # per-call authorization, even inside an approved script
                calls.append((name, "DENIED"))
                raise PermissionError(f"{name} is not approved for this run")
            r = await host.call(name, kwargs)
            calls.append((name, "error" if r.is_error else "ok"))
            if r.is_error:
                raise ScriptToolError(r.content[0].text)
            sc = r.structured_content
            return sc["result"] if sc and set(sc) == {"result"} else (sc or r.content[0].text)
        return stub

    try:
        check_script(code)
        env = {"__builtins__": SAFE_BUILTINS, "ToolError": ScriptToolError,
               "print": lambda *a: out.append(" ".join(map(str, a)))}
        env.update({d["name"]: make_stub(d["name"]) for d in travel_defs})
        body = "\n".join("    " + line for line in code.splitlines())
        exec(compile(f"async def _script():\n{body}\n", "<model-code>", "exec"), env)
        await asyncio.wait_for(env["_script"](), timeout)
        status = "ok"
    except Exception as exc:                            # surface uncaught errors so the model can self-correct
        status = f"{type(exc).__name__}: {exc}"
    text = "\n".join(out)
    return {"status": status, "output": text[:max_output] + ("…[truncated]" if len(text) > max_output else ""),
            "calls": calls}


print(await run_script("import os\nprint(os.environ)", grant=set()))
print(await run_script("print(().__class__.__mro__)", grant=set()))
print(await run_script("x = await weather__forecast(city='Paris', date='2026-06-15')", grant={"weather__forecast"}))
print(await run_script('''
try:
    sky = await weather__forecast(city="Paris", date="2026-06-15")
except ToolError as err:
    sky = f"unavailable ({err})"
print("Paris:", sky)
''', grant={"weather__forecast"}))

看看这四个结果：两个脚本在运行前就被拒绝；一个未捕获的工具错误被作为脚本的结果呈现（让模型可以自我纠正）；还有一个脚本用 `try`/`except` 自行处理了 `isError`。

### 8. 直接工具调用与代码模式的对比
任务：<em>“找出 2026-06-15 从 NYC 到 BCN 最便宜的直飞航班、一家低于 180 美元的酒店，以及当天的天气。”</em> 先用直接方式：三次工具调用，每个完整结果都要回到模型。再用代码模式（Code Mode）：模型编写一个脚本，数据留在执行器中，只返回一行摘要。

In [ ]:
TASK_TOOLS = [("flights__search", {"origin": "NYC", "destination": "BCN", "date": "2026-06-15"}),
              ("hotels__search", {"city": "Barcelona", "max_price": 180}),
              ("weather__forecast", {"city": "Barcelona", "date": "2026-06-15"})]

direct_tokens, model_calls = 0, 1                 # the first model call decides to use tools
for name, args in TASK_TOOLS:                     # one tool call per turn, as a cautious model would
    r = await host.call(name, args)
    payload = r.structured_content or r.content[0].text
    direct_tokens += est_tokens(payload)
    model_calls += 1
    print(f"direct: {name:<18} -> ~{est_tokens(payload):>5} tokens into context")
print(f"direct total: {model_calls} model calls, ~{direct_tokens} tokens of tool results in context\n")

MODEL_SCRIPT = '''
flights = await flights__search(origin="NYC", destination="BCN", date="2026-06-15")
nonstop = [f for f in flights if f["stops"] == 0]
best = min(nonstop, key=lambda f: f["price"])
hotels = await hotels__search(city="Barcelona", max_price=180)
hotel = max(hotels, key=lambda h: h["rating"])
sky = await weather__forecast(city="Barcelona", date="2026-06-15")
print(f"{len(flights)} flights checked; cheapest non-stop {best['id']} at {best['depart']} for ${best['price']}; "
      f"best hotel under $180: {hotel['name']} (${hotel['price']}, {hotel['rating']}*); weather: {sky}")
'''
run = await run_script(MODEL_SCRIPT, grant={"flights__search", "hotels__search", "weather__forecast"})
print("code mode broker log:", run["calls"])
print("code mode output    :", run["output"])
print(f"code mode total: 2 model calls, ~{est_tokens(MODEL_SCRIPT) + est_tokens(run['output'])} tokens "
      f"(script + output) in context")

接下来是文档中的安全规则：批准脚本<strong>不等于</strong>一揽子批准。下面的授权涵盖了搜索，但不包括预订，所以代理在运行时拒绝了这个有副作用的调用，该错误成为脚本的结果。

In [ ]:
greedy = '''
hotels = await hotels__search(city="Barcelona", max_price=180)
print(await hotels__book(name=hotels[0]["name"], nights=7))
'''
print(json.dumps(await run_script(greedy, grant={"hotels__search"}), indent=2))

### 9. 可选：使用渐进式发现的 Claude
设置 `ANTHROPIC_API_KEY` 之后，Claude 只拿到三个元工具（Meta-tool），必须自己去发现旅行相关的工具。循环与第 5.2 课相同，只有 `tools` 数组变了。`tools` 数组在每一轮都保持完全一致，而这正是让提供方的提示缓存（Prompt Caching）保持命中的关键。

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

    llm = anthropic.Anthropic()
    messages = [{"role": "user", "content": "I fly NYC to Barcelona on 2026-06-15. What's the cheapest "
                 "non-stop flight, and what's the weather that day? Discover the tools you need first."}]
    response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, tools=META_TOOLS, messages=messages)
    while response.stop_reason == "tool_use":
        messages.append({"role": "assistant", "content": response.content})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                try:
                    out, is_error = await run_meta_tool(block.name, block.input), False
                except Exception as exc:
                    out, is_error = f"{type(exc).__name__}: {exc}", True
                print(f"Claude -> {block.name}({json.dumps(block.input)[:120]})")
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": json.dumps(out)[:4000], "is_error": is_error})
        messages.append({"role": "user", "content": results})
        response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, tools=META_TOOLS,
                                       messages=messages)
    print("Claude:", "".join(b.text for b in response.content if b.type == "text"))
else:
    print("skipped: set ANTHROPIC_API_KEY to watch Claude search_tools -> get_tool_details -> call_tool.")

await host.close()

### 动手练习
1. 把 `search_tools` 中的关键词打分器换成更聪明的实现（例如通过简单的词频权重实现 BM25，或加入“hotel”/“room”这类同义词）。真实用户的哪些查询仍然会失败？
2. 在渐进式流程中加入 `disable(alias)`：在天气调用之后断开 `weather`，并展示 `host.clients` 随之缩小。为什么文档说要把断开连接视为对话边界级别的操作？
3. 给 weather 服务器设置 `CacheHint(ttl_ms=0)`，然后重新运行第 5 节。宿主现在会发送多少次 `tools/list` 请求？这是 bug 吗？
4. 为执行器编写一个脚本：遍历三个日期，对每个日期调用 `flights__search`，只打印每天最便宜的票价。统计与直接调用相比节省了多少令牌。